# Phase 3 · Deliverable 4: The Benchmarking Script (Business Validation)

**Project:** Optimizing Delivery ETAs with Graph-Based Network Intelligence  
**Author:** Lead Data Science Team — Delhivery ETA Optimisation

## Purpose

Ingests held-out test-set predictions from the three Phase 3 model architectures (Baseline XGBoost, Hybrid Node2Vec+XGBoost, GraphSAGE GNN), computes standard ML error metrics (MAE, RMSE) alongside a custom business SLA adherence metric (<15% error), and outputs an executive-ready comparison table proving the superiority of graph-enhanced architectures.

## Expected Input Files

Place in `data/processed/` or `results/`:
- `baseline_predictions.csv`
- `hybrid_predictions.csv`
- `graphsage_predictions.csv`

Each CSV must contain at minimum:
- `actual_eta` : float — ground truth delivery time (minutes)
- `predicted_eta` : float — model's predicted delivery time (minutes)

## Imports

In [1]:
import os
import sys
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")

# tabulate is the cleanest way to render executive tables in the terminal.
# We provide a graceful fallback to pandas .to_string() if it is not installed.
try:
    from tabulate import tabulate
    TABULATE_AVAILABLE = True
except ImportError:
    TABULATE_AVAILABLE = False

## Configuration

Adjust paths to match your local repo layout.

In [2]:
# Priority search order: results/ first, then data/processed/
SEARCH_DIRS: List[Path] = [
    Path("results"),
    Path("data/processed"),
    Path("."),                    # fallback: same directory as this script
]

MODEL_FILE_MAP: Dict[str, str] = {
    "Baseline (XGBoost)":            "baseline_predictions.csv",
    "Hybrid (Node2Vec + XGBoost)":   "hybrid_predictions.csv",
    "Deep GNN (GraphSAGE)":          "graphsage_predictions.csv",
}

# Column names expected inside each CSV
COL_ACTUAL    = "actual_eta"
COL_PREDICTED = "predicted_eta"

# Business SLA threshold (15 %)
SLA_THRESHOLD_PCT = 0.15

# Minimum number of valid rows required before we trust the metrics
MIN_VALID_ROWS = 100

## 1 · Data Ingestion & Validation

In [3]:
def _locate_file(filename: str, search_dirs: List[Path]) -> Optional[Path]:
    """
    Search candidate directories in order and return the first path at which
    `filename` exists, or None if not found anywhere.

    Parameters
    ----------
    filename    : str        — bare filename (e.g. 'baseline_predictions.csv')
    search_dirs : list[Path] — directories to probe, tried in order

    Returns
    -------
    Path | None
    """
    for directory in search_dirs:
        candidate = directory / filename
        if candidate.exists():
            return candidate
    return None


def load_predictions(
    filename: str,
    search_dirs: List[Path] = SEARCH_DIRS,
    col_actual: str    = COL_ACTUAL,
    col_predicted: str = COL_PREDICTED,
) -> pd.DataFrame:
    """
    Locate and load a model predictions CSV, then perform schema validation
    and basic row-level sanity cleaning.

    Steps performed:
        1. Locate the file across candidate directories.
        2. Load with pandas, keeping only the two required columns.
        3. Coerce both columns to float (catches string-encoded numerics).
        4. Drop rows where either column is NaN or non-finite.
        5. Drop rows where actual_eta == 0 (prevents division-by-zero in
           the SLA metric calculation — see `calculate_sla_adherence`).
        6. Warn if the surviving row count is below MIN_VALID_ROWS.

    Parameters
    ----------
    filename      : str       — CSV filename to locate
    search_dirs   : list[Path] — directories to search (default: SEARCH_DIRS)
    col_actual    : str       — column name for ground-truth values
    col_predicted : str       — column name for predicted values

    Returns
    -------
    pd.DataFrame
        Cleaned DataFrame with exactly two columns: [col_actual, col_predicted]

    Raises
    ------
    FileNotFoundError  — if the file cannot be found in any search directory
    ValueError         — if required columns are missing from the CSV
    """
    path = _locate_file(filename, search_dirs)
    if path is None:
        searched = [str(d) for d in search_dirs]
        raise FileNotFoundError(
            f"Could not locate '{filename}' in any of: {searched}\n"
            f"  ↳ Place the file in one of those directories and re-run."
        )

    df = pd.read_csv(path)

    # ── Column presence check ──────────────────────────────────────────────
    missing_cols = [c for c in [col_actual, col_predicted] if c not in df.columns]
    if missing_cols:
        raise ValueError(
            f"File '{path}' is missing required column(s): {missing_cols}\n"
            f"  Available columns: {list(df.columns)}"
        )

    df = df[[col_actual, col_predicted]].copy()

    # ── Type coercion ──────────────────────────────────────────────────────
    df[col_actual]    = pd.to_numeric(df[col_actual],    errors="coerce")
    df[col_predicted] = pd.to_numeric(df[col_predicted], errors="coerce")

    # ── Drop NaN / infinite rows ──────────────────────────────────────────
    n_before = len(df)
    df = df.replace([np.inf, -np.inf], np.nan).dropna(
        subset=[col_actual, col_predicted]
    )
    n_nan_dropped = n_before - len(df)

    # ── Drop rows where actual_eta == 0 (mathematically unsafe) ──────────
    n_before_zero = len(df)
    df = df[df[col_actual] != 0].copy()
    n_zero_dropped = n_before_zero - len(df)

    # ── Reporting ─────────────────────────────────────────────────────────
    total_dropped = n_nan_dropped + n_zero_dropped
    if total_dropped > 0:
        print(
            f"  ⚠  {filename}: dropped {n_nan_dropped} NaN/Inf rows and "
            f"{n_zero_dropped} zero-ETA rows  →  {len(df):,} valid rows remain"
        )

    if len(df) < MIN_VALID_ROWS:
        print(
            f"  ⚠  WARNING: only {len(df)} valid rows found in '{filename}'. "
            f"Metrics may not be statistically reliable."
        )

    return df.reset_index(drop=True)


def load_all_models(
    model_file_map: Dict[str, str] = MODEL_FILE_MAP,
    search_dirs: List[Path]        = SEARCH_DIRS,
) -> Dict[str, pd.DataFrame]:
    """
    Load prediction DataFrames for all models defined in `model_file_map`.

    Parameters
    ----------
    model_file_map : dict[str, str]  — {architecture_label: csv_filename}
    search_dirs    : list[Path]      — directories to probe for each file

    Returns
    -------
    dict[str, pd.DataFrame]
        Keys are the architecture labels; values are cleaned DataFrames.
        Models whose files cannot be found are skipped with a warning.
    """
    loaded: Dict[str, pd.DataFrame] = {}
    print("─" * 60)
    print("  LOADING PREDICTION FILES")
    print("─" * 60)

    for label, filename in model_file_map.items():
        try:
            df = load_predictions(filename, search_dirs)
            loaded[label] = df
            print(f"  ✅  {label:<35} → {len(df):,} rows  ({filename})")
        except FileNotFoundError as e:
            print(f"  ❌  {label:<35} → SKIPPED  ({e})")
        except ValueError as e:
            print(f"  ❌  {label:<35} → SCHEMA ERROR  ({e})")

    print()
    return loaded

## 2 · Standard Metrics — MAE & RMSE

In [4]:
def calculate_standard_metrics(
    df: pd.DataFrame,
    col_actual: str    = COL_ACTUAL,
    col_predicted: str = COL_PREDICTED,
) -> Dict[str, float]:
    """
    Compute Mean Absolute Error (MAE) and Root Mean Squared Error (RMSE)
    using scikit-learn's battle-tested implementations.

    These are the primary regression quality metrics used to compare
    model accuracy in units of minutes.

    Parameters
    ----------
    df            : pd.DataFrame — cleaned predictions DataFrame
    col_actual    : str          — column name for ground-truth ETAs
    col_predicted : str          — column name for model predictions

    Returns
    -------
    dict with keys:
        'mae'  : float — Mean Absolute Error (minutes)
        'rmse' : float — Root Mean Squared Error (minutes)

    Notes
    -----
    RMSE penalises large errors more heavily than MAE because residuals are
    squared before averaging. A large gap between MAE and RMSE indicates the
    model is producing occasional catastrophic outlier predictions, which is
    particularly harmful in SLA-sensitive logistics operations.
    """
    y_true = df[col_actual].values
    y_pred = df[col_predicted].values

    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))

    return {
        "mae":  round(mae,  4),
        "rmse": round(rmse, 4),
    }

## 3 · Custom Business Metric — 15% SLA Adherence

In [5]:
def calculate_sla_adherence(
    df: pd.DataFrame,
    col_actual: str    = COL_ACTUAL,
    col_predicted: str = COL_PREDICTED,
    threshold: float   = SLA_THRESHOLD_PCT,
) -> float:
    """
    Calculate the custom logistics SLA adherence metric.

    Definition
    ----------
    The **percentage of trips** for which the absolute prediction error
    is at most `threshold` × actual_eta:

        SLA Adherence (%) =
            mean( |predicted_eta − actual_eta| ≤ threshold × actual_eta ) × 100

    This directly mirrors the operational KPI: "What fraction of our shipments
    do we commit an ETA to within 15% of the real delivery time?"

    Mathematical Safety
    -------------------
    Rows where actual_eta == 0 create division-by-zero risk. These are
    removed during ingestion (`load_predictions`) so this function always
    receives a clean DataFrame. An additional guard is included here for
    defensive completeness in case this function is called in isolation.

    Parameters
    ----------
    df            : pd.DataFrame — cleaned predictions DataFrame (actual_eta != 0)
    col_actual    : str          — column name for ground-truth ETAs
    col_predicted : str          — column name for model predictions
    threshold     : float        — SLA boundary as a fraction (default: 0.15 → 15%)

    Returns
    -------
    float
        SLA adherence as a percentage value (0.0 – 100.0).
    """
    # Defensive guard: drop zero-actual rows if any slipped through ingestion
    safe_df = df[df[col_actual] != 0].copy()
    if len(safe_df) < len(df):
        dropped = len(df) - len(safe_df)
        print(f"  ⚠  SLA metric: dropped {dropped} zero-ETA row(s) for safety.")

    y_true = safe_df[col_actual].values
    y_pred = safe_df[col_predicted].values

    # Core SLA formula
    abs_error        = np.abs(y_pred - y_true)
    sla_bound        = threshold * y_true
    within_sla       = abs_error <= sla_bound

    sla_adherence_pct = within_sla.mean() * 100.0
    return round(sla_adherence_pct, 2)

## 4 · Full Evaluation Pipeline — Per-Model Orchestration

In [6]:
def evaluate_model(
    df: pd.DataFrame,
    label: str,
    col_actual: str    = COL_ACTUAL,
    col_predicted: str = COL_PREDICTED,
    sla_threshold: float = SLA_THRESHOLD_PCT,
) -> Dict[str, object]:
    """
    Run the complete evaluation suite for a single model's predictions.

    Calls:
        - `calculate_standard_metrics`  →  MAE, RMSE
        - `calculate_sla_adherence`     →  % trips within 15% error

    Parameters
    ----------
    df            : pd.DataFrame — cleaned predictions DataFrame
    label         : str          — human-readable model architecture name
    col_actual    : str          — column name for ground-truth ETAs
    col_predicted : str          — column name for model predictions
    sla_threshold : float        — SLA boundary fraction (default: 0.15)

    Returns
    -------
    dict with keys:
        'model'          : str   — architecture label
        'n_samples'      : int   — number of evaluated rows
        'mae'            : float — MAE in minutes
        'rmse'           : float — RMSE in minutes
        'sla_adherence'  : float — % predictions within SLA threshold
    """
    standard = calculate_standard_metrics(df, col_actual, col_predicted)
    sla      = calculate_sla_adherence(df, col_actual, col_predicted, sla_threshold)

    return {
        "model":         label,
        "n_samples":     len(df),
        "mae":           standard["mae"],
        "rmse":          standard["rmse"],
        "sla_adherence": sla,
    }


def evaluate_all_models(
    predictions: Dict[str, pd.DataFrame],
    col_actual: str    = COL_ACTUAL,
    col_predicted: str = COL_PREDICTED,
    sla_threshold: float = SLA_THRESHOLD_PCT,
) -> List[Dict[str, object]]:
    """
    Evaluate every loaded model and return a list of result dictionaries,
    ordered to match the input dictionary (insertion order preserved in
    Python 3.7+, which respects the Baseline → Hybrid → GNN progression).

    Parameters
    ----------
    predictions   : dict[str, pd.DataFrame] — output of `load_all_models`
    col_actual    : str                     — actual ETA column name
    col_predicted : str                     — predicted ETA column name
    sla_threshold : float                   — SLA boundary fraction

    Returns
    -------
    list[dict]  — one result dict per model (see `evaluate_model` for schema)
    """
    results = []
    for label, df in predictions.items():
        result = evaluate_model(df, label, col_actual, col_predicted, sla_threshold)
        results.append(result)
    return results

## 5 · Output & Business Presentation

In [7]:
def _render_tabulate(results: List[Dict]) -> str:
    """Render the results table using the tabulate library (preferred)."""
    headers = [
        "Model Architecture",
        "Samples",
        "MAE (min)",
        "RMSE (min)",
        f"SLA Adherence (<{int(SLA_THRESHOLD_PCT*100)}% Error)",
    ]
    rows = [
        [
            r["model"],
            f"{r['n_samples']:,}",
            f"{r['mae']:.2f}",
            f"{r['rmse']:.2f}",
            f"{r['sla_adherence']:.2f}%",
        ]
        for r in results
    ]
    return tabulate(rows, headers=headers, tablefmt="fancy_grid", numalign="right")


def _render_pandas(results: List[Dict]) -> str:
    """Fallback renderer using pandas console output (no tabulate dependency)."""
    df = pd.DataFrame(results).rename(columns={
        "model":         "Model Architecture",
        "n_samples":     "Samples",
        "mae":           "MAE (min)",
        "rmse":          "RMSE (min)",
        "sla_adherence": f"SLA Adherence (<{int(SLA_THRESHOLD_PCT*100)}% Error)",
    })
    df["MAE (min)"]  = df["MAE (min)"].map(lambda x: f"{x:.2f}")
    df["RMSE (min)"] = df["RMSE (min)"].map(lambda x: f"{x:.2f}")
    df[f"SLA Adherence (<{int(SLA_THRESHOLD_PCT*100)}% Error)"] = \
        df[f"SLA Adherence (<{int(SLA_THRESHOLD_PCT*100)}% Error)"].map(lambda x: f"{x:.2f}%")
    df["Samples"] = df["Samples"].map(lambda x: f"{int(x.replace(',', '')):,}"
                                       if isinstance(x, str) else f"{x:,}")
    return df.to_string(index=False)


def print_comparison_table(results: List[Dict]) -> None:
    """
    Print an executive-ready benchmarking comparison table to stdout.

    Selects the tabulate renderer if the library is available, otherwise
    falls back to a formatted pandas string representation.

    Additionally prints:
        - A business narrative identifying the best model per metric.
        - A delta analysis showing improvement over the baseline.

    Parameters
    ----------
    results : list[dict] — output of `evaluate_all_models`
    """
    if not results:
        print("  ⚠  No results to display — no models were successfully evaluated.")
        return

    print("\n" + "=" * 70)
    print("  PHASE 3 MODEL BENCHMARKING REPORT")
    print(f"  SLA Threshold: predictions within ±{int(SLA_THRESHOLD_PCT*100)}% of actual ETA")
    print("=" * 70)

    if TABULATE_AVAILABLE:
        print(_render_tabulate(results))
    else:
        print(_render_pandas(results))
        print("\n  [Install tabulate for a prettier table: pip install tabulate]")

    # ── Business narrative & delta analysis ───────────────────────────────
    if len(results) > 1:
        _print_business_narrative(results)


def _print_business_narrative(results: List[Dict]) -> None:
    """
    Print a structured narrative comparing models, identifying the winner
    on each metric and quantifying the lift over the baseline.

    Assumes the first result in `results` is the Baseline model.
    """
    baseline = results[0]
    others   = results[1:]

    best_mae  = min(results, key=lambda r: r["mae"])
    best_rmse = min(results, key=lambda r: r["rmse"])
    best_sla  = max(results, key=lambda r: r["sla_adherence"])

    print("\n" + "─" * 70)
    print("  BUSINESS VERDICT")
    print("─" * 70)

    print(f"\n  🏆  Best MAE   → {best_mae['model']:<38} ({best_mae['mae']:.2f} min)")
    print(f"  🏆  Best RMSE  → {best_rmse['model']:<38} ({best_rmse['rmse']:.2f} min)")
    print(f"  🏆  Best SLA   → {best_sla['model']:<38} ({best_sla['sla_adherence']:.2f}%)")

    print("\n  DELTA vs BASELINE  ({})".format(baseline["model"]))
    print("  " + "─" * 50)

    for result in others:
        mae_delta  = baseline["mae"]  - result["mae"]          # positive = improvement
        rmse_delta = baseline["rmse"] - result["rmse"]         # positive = improvement
        sla_delta  = result["sla_adherence"] - baseline["sla_adherence"]  # positive = improvement

        mae_pct  = (mae_delta  / baseline["mae"])           * 100  if baseline["mae"]           != 0 else 0.0
        rmse_pct = (rmse_delta / baseline["rmse"])          * 100  if baseline["rmse"]          != 0 else 0.0
        sla_pct  = (sla_delta  / baseline["sla_adherence"]) * 100  if baseline["sla_adherence"] != 0 else 0.0

        print(f"\n  {result['model']}")
        _fmt_delta("MAE",           mae_delta,  mae_pct,  unit="min",  lower_is_better=True)
        _fmt_delta("RMSE",          rmse_delta, rmse_pct, unit="min",  lower_is_better=True)
        _fmt_delta("SLA Adherence", sla_delta,  sla_pct,  unit="%",    lower_is_better=False)

    print("\n" + "=" * 70)
    print("  ✅  Evaluation complete.")
    print("=" * 70 + "\n")


def _fmt_delta(
    metric: str,
    absolute_delta: float,
    pct_delta: float,
    unit: str,
    lower_is_better: bool,
) -> None:
    """
    Pretty-print a single metric delta line with directional emoji.

    Parameters
    ----------
    metric          : str   — metric name (e.g. 'MAE')
    absolute_delta  : float — raw difference (positive = better for MAE/RMSE)
    pct_delta       : float — percentage improvement over baseline
    unit            : str   — display unit ('min' or '%')
    lower_is_better : bool  — controls the emoji direction logic
    """
    if lower_is_better:
        improved = absolute_delta > 0
    else:
        improved = absolute_delta > 0

    icon   = "✅" if improved else "❌"
    sign   = "+" if improved else ""
    change = "better" if improved else "worse"

    print(
        f"    {icon}  {metric:<18}: "
        f"{sign}{absolute_delta:+.2f} {unit}  "
        f"({sign}{pct_delta:.1f}% {change} than baseline)"
    )

## 6 · Additional Diagnostic Output (optional, aids debugging)

In [8]:
def print_error_distribution_summary(
    predictions: Dict[str, pd.DataFrame],
    col_actual: str    = COL_ACTUAL,
    col_predicted: str = COL_PREDICTED,
) -> None:
    """
    For each model, print a concise error distribution summary:
    the 10th, 25th, 50th, 75th, 90th, and 95th percentiles of the
    absolute prediction error in minutes.

    Useful for diagnosing whether one model is better in the tail (95th+)
    versus the median — which has different operational implications.

    Parameters
    ----------
    predictions   : dict[str, pd.DataFrame]
    col_actual    : str
    col_predicted : str
    """
    print("\n" + "─" * 70)
    print("  ERROR DISTRIBUTION SUMMARY  (absolute error in minutes)")
    print("─" * 70)

    pcts = [10, 25, 50, 75, 90, 95]
    header = f"  {'Model':<38}  " + "  ".join(f"P{p:02d}" for p in pcts)
    print(header)
    print("  " + "─" * (len(header) - 2))

    for label, df in predictions.items():
        abs_err = (df[col_predicted] - df[col_actual]).abs()
        quantiles = abs_err.quantile([p / 100 for p in pcts]).values
        row = f"  {label:<38}  " + "  ".join(f"{q:5.1f}" for q in quantiles)
        print(row)

    print()

## Main Entrypoint

End-to-end orchestration:
1. Load all three model prediction CSVs.
2. Evaluate each model (MAE, RMSE, SLA adherence).
3. Print the executive comparison table with business narrative.
4. Print supplementary error distribution diagnostics.

In [9]:
def main() -> None:
    """
    End-to-end orchestration:

    1. Load all three model prediction CSVs.
    2. Evaluate each model (MAE, RMSE, SLA adherence).
    3. Print the executive comparison table with business narrative.
    4. Print supplementary error distribution diagnostics.

    Exit codes:
        0 — success (at least one model evaluated)
        1 — failure (no models could be loaded)
    """
    print("\n" + "═" * 70)
    print("  DELHIVERY ETA OPTIMISATION — PHASE 3 EVALUATION PIPELINE")
    print("  Baseline  |  Node2Vec Hybrid  |  GraphSAGE GNN")
    print("═" * 70 + "\n")

    # ── Step 1: Ingest ─────────────────────────────────────────────────────
    predictions = load_all_models(MODEL_FILE_MAP, SEARCH_DIRS)

    if not predictions:
        print(
            "\n  ❌  FATAL: No prediction files could be loaded.\n"
            "     Ensure the CSVs are placed in one of these directories:\n"
            + "".join(f"       • {d}\n" for d in SEARCH_DIRS)
        )
        sys.exit(1)

    # ── Step 2: Evaluate ───────────────────────────────────────────────────
    results = evaluate_all_models(predictions)

    # ── Step 3: Print executive table + narrative ──────────────────────────
    print_comparison_table(results)

    # ── Step 4: Supplementary distribution diagnostics ────────────────────
    print_error_distribution_summary(predictions)


# Allow the script to be both run directly and imported as a module.
# When imported, callers can invoke load_all_models / evaluate_all_models /
# print_comparison_table independently (e.g. from a Jupyter notebook).
if __name__ == "__main__":
    main()


══════════════════════════════════════════════════════════════════════
  DELHIVERY ETA OPTIMISATION — PHASE 3 EVALUATION PIPELINE
  Baseline  |  Node2Vec Hybrid  |  GraphSAGE GNN
══════════════════════════════════════════════════════════════════════

────────────────────────────────────────────────────────────
  LOADING PREDICTION FILES
────────────────────────────────────────────────────────────
  ❌  Baseline (XGBoost)                  → SKIPPED  (Could not locate 'baseline_predictions.csv' in any of: ['results', 'data/processed', '.']
  ↳ Place the file in one of those directories and re-run.)
  ❌  Hybrid (Node2Vec + XGBoost)         → SKIPPED  (Could not locate 'hybrid_predictions.csv' in any of: ['results', 'data/processed', '.']
  ↳ Place the file in one of those directories and re-run.)
  ❌  Deep GNN (GraphSAGE)                → SKIPPED  (Could not locate 'graphsage_predictions.csv' in any of: ['results', 'data/processed', '.']
  ↳ Place the file in one of those directories and 

SystemExit: 1

## Run Evaluation

In [ ]:
main()